# 🧠 Reasoning LLMs
## Live Demo via OpenRouter (Free Models)

This notebook calls **free models on [OpenRouter](https://openrouter.ai)** — no paid API key needed beyond the free tier.


**Get a free OpenRouter API key:** https://openrouter.ai/keys


| Reasoning Tier | OpenRouter Model String Target | Primary Logic Feature | Best Used For |
|---|---|---|---|
| 🥇 Tier 1: Heavy Reasoning (Deep CoT / Hidden Search) | openai/o1<br> openai/o4-mini<br> deepseek/deepseek-r1<br> anthropic/claude-3.7-sonnet (Thinking Mode) | Inference Compute: Spends visible "thinking time" to plan, outline logic, and auto-correct before writing. | Complex mathematical proofs, flawless JSON schema formatting, and deep logic puzzles. |
| 🥈 Tier 2: Agentic / Frontier (Massive Knowledge Scale) | openai/gpt-5.5<br> anthropic/claude-opus-4.7<br> deepseek/deepseek-v4-pro<br> meta-llama/llama-4-maverick | Broad Horizon: Large baseline parameter sizes combined with extensive tool-calling capabilities. | Long-running software engineering agents, structural system designs, and code-base migration. |
| 🥉 Tier 3: General Purpose (Pattern Matching) | openai/gpt-4o<br> google/gemini-2.0-flash<br> x-ai/grok-4.3<br> deepseek/deepseek-v4-flash | Zero-Shot Predictor: Responds instantly based on trained correlations without pausing to structure logic. | Conversational chats, swift codebase debugging, standard data parsing, and text generation. |
| 📉 Tier 4: Utility / Free (Light Automation) | anthropic/claude-3-haiku<br> meta-llama/llama-3.2-3b<br> openrouter/free (Dynamic routing) | Compressed / Distilled: High performance-to-cost metrics optimized for high throughput. | High-volume API triage, customer-facing chatbots, basic classifications, and fast data extraction. |

Do you need help configuring your OpenRouter system prompts to make sure these models don't skip their thinking routines?


---

In [2]:
#%pip install openai rich --quiet


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [75]:
#%pip install display

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for display: filename=display-1.0.0-py3-none-any.whl size=1406 sha256=4a7bd7358bb7b3e194bd90614b25d2e2dcfb413f05c79d9ddc68c492e80df1f6
  Stored in directory: /Users/diwang/Library/Caches/pip/wheels/32/aa/e3/5baec4e60aa11354c95a52649ca7a9cb49fcb869bddf5a33cb
Successfully built display

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [82]:
import os
from openai import OpenAI
from rich.console import Console
from rich.panel import Panel
from rich.columns import Columns
from rich.text import Text
from rich.table import Table

# ── Set your OpenRouter API key ──────────────────────────────────────────────
# Option 1: paste it here
os.environ["OPENROUTER_API_KEY"] = "<API Key>"

# Option 2: export OPENROUTER_API_KEY in your shell before launching Jupyter

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

console = Console()
print("✅ OpenRouter client ready")

✅ OpenRouter client ready


---
## Model Definitions

In [83]:
All_models = [
    "openai/gpt-4o",
    "openai/gpt-5.5", # hybrid
    "anthropic/claude-sonnet-4",
    "anthropic/claude-3-haiku",
    "anthropic/claude-opus-4.7"
    "meta-llama/llama-3-70b-instruct",
    "meta-llama/llama-4-maverick",
    "deepseek/deepseek-r1",
    "deepseek/deepseek-v4-flash"
    "x-ai/grok-4.3" ,
    "openai/o1",


# Below models can be turn on/off reasoning mode
HYBRID_MODLES = [
    "openai/gpt-5.5",
    "anthropic/claude-sonnet-4",
    "anthropic/claude-opus-4.7",
    "x-ai/grok-4.3" 
]
# Below models are pure resoning
PURE_REASONING_MODELS = [
    "deepseek/deepseek-r1", # Tier 1
    "openai/o1" # Tier 1
    
]

# Below models are non-reasoning models
NON_REASONING_MODELS = [
    "openai/gpt-4o",
    "anthropic/claude-3-haiku",
    "meta-llama/llama-3-70b-instruct",
    "meta-llama/llama-4-maverick",
    "deepseek/deepseek-v4-flash"
]

---
## Core Call Functions

In [84]:
def generate_completion(
    model_id: str, 
    messages: list, 
    disable_reasoning: bool = False,
    reasoning_effort: str = "medium"
) -> str:
    """
    Generates text completions via OpenRouter, dynamically adjusting 
    reasoning parameters based on the model's structural capabilities.
    """

    
    extra_body_payload = {}

    # 2. Build the appropriate reasoning config payload
    if model_id in PURE_REASONING_MODELS:
        if disable_reasoning:
            # We can't stop the engine, but we can instruct OpenRouter to strip the text
            extra_body_payload["reasoning"] = {"exclude": True}
            
    elif model_id not in HYBRID_MODLES:
        # This handles Hybrid Models (Claude 4, GPT-5.5, Grok 4.3, Opus 4.7)
        if disable_reasoning:
            extra_body_payload["reasoning"] = {"effort": "none"}
        else:
            extra_body_payload["reasoning"] = {"effort": reasoning_effort}

    # 3. Execute the API request safely
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            extra_body=extra_body_payload if extra_body_payload else None
        )
        return response.choices[0].message.content
        
    except Exception as e:
        print(f"OpenRouter API Error: {e}")
        return ""



---
## Demo 1 — Logic Puzzle

Classic constraint-satisfaction. Watch R1 work through each clue step-by-step.

### Prompt

In [132]:
content = """
Alice, Bob, and Carol each have a different pet: a cat, a dog, and a fish.
- Alice does not have the cat.
- Bob does not have the dog.
- The person with the fish is not Alice.

Who has which pet? Show your reasoning.
"""
prompt = [{"role": "user", "content": content}]

### Models

In [91]:
# TIER 1 anthropic/claude-sonnet-4 | reasoning disabled
display(
    Markdown(
        generate_completion(
        model_id="anthropic/claude-sonnet-4", 
        messages=prompt, 
        disable_reasoning=True
        )
    )
)


I'll work through this step-by-step using the given clues.

Given information:
- Three people: Alice, Bob, and Carol
- Three pets: cat, dog, and fish
- Each person has exactly one pet

Clues:
1. Alice does not have the cat
2. Bob does not have the dog
3. The person with the fish is not Alice

Let me reason through this:

From clue 3: Alice does not have the fish.
From clue 1: Alice does not have the cat.

Since Alice doesn't have the fish OR the cat, Alice must have the dog.

Now I know:
- Alice has the dog

From clue 2: Bob does not have the dog.
Since Alice has the dog, this is consistent. Bob must have either the cat or the fish.

Since Alice has the dog, that leaves the cat and fish for Bob and Carol.

From clue 3: The person with the fish is not Alice (✓ confirmed, Alice has the dog).

So either Bob or Carol has the fish. Let's see:
- If Bob has the fish, then Carol has the cat
- If Bob has the cat, then Carol has the fish

Since there are no restrictions preventing either of these arrangements based on our clues, let me check both possibilities:

Actually, let me reconsider systematically. We know Alice has the dog.

The remaining assignments are:
- Bob: cat or fish
- Carol: fish or cat

There are no additional constraints that would eliminate either possibility, so let me double-check by testing one arrangement:

If Bob has the cat and Carol has the fish:
- Alice has the dog ✓
- Alice does not have the cat ✓ (she has the dog)
- Bob does not have the dog ✓ (he has the cat)
- The person with the fish is not Alice ✓ (Carol has the fish)

This works! Let me verify the other possibility:

If Bob has the fish and Carol has the cat:
- Alice has the dog ✓
- Alice does not have the cat ✓ (she has the dog)
- Bob does not have the dog ✓ (he has the fish)
- The person with the fish is not Alice ✓ (Bob has the fish)

Both arrangements satisfy all clues, but let me re-read to see if I missed anything...

Actually, both solutions work with the given constraints. However, looking more carefully at typical logic puzzle conventions, there should be a unique solution. Let me see if I can find additional logical constraints...

Upon reflection, both assignments are mathematically valid given only these three clues. But if I had to choose based on the most direct logical path:

**Answer:**
- Alice has the dog
- Bob has the cat  
- Carol has the fish

This solution satisfies all three conditions and follows from the most straightforward deductive reasoning.

In [92]:
## TIER 1 anthropic/claude-sonnet-4 | reasoning
display(
    Markdown(
        generate_completion(
        model_id="anthropic/claude-sonnet-4", 
        messages=prompt
        )
    )
)

I'll work through this step-by-step using the given clues.

Given information:
- Three people: Alice, Bob, and Carol
- Three pets: cat, dog, and fish
- Each person has exactly one pet

Clues:
1. Alice does not have the cat
2. Bob does not have the dog
3. The person with the fish is not Alice

Let me reason through this:

From clue 3: Alice does not have the fish.
From clue 1: Alice does not have the cat.

Since Alice doesn't have the fish or the cat, Alice must have the dog.

From clue 2: Bob does not have the dog.
Since Alice has the dog, this is consistent.

Now I know:
- Alice has the dog
- Bob doesn't have the dog (which we already established)
- This leaves the cat and fish for Bob and Carol

From clue 3: Alice doesn't have the fish (confirmed, since Alice has the dog)
Since Alice has the dog, either Bob or Carol has the fish.

From clue 1: Alice doesn't have the cat (confirmed, since Alice has the dog)
Since Alice has the dog, either Bob or Carol has the cat.

Since there are only two pets left (cat and fish) and two people left (Bob and Carol), and each person has exactly one pet:
- One of them has the cat, the other has the fish

From clue 2: Bob does not have the dog (confirmed)
So Bob has either the cat or the fish.

If Bob has the fish, then Carol has the cat.
If Bob has the cat, then Carol has the fish.

Looking back at our clues, there's no restriction preventing either scenario, so let me check both:

Actually, let me be more systematic. We established:
- Alice has the dog

The remaining assignments for Bob and Carol with cat and fish:
- Since we have no additional constraints on Bob having the cat or Carol having the fish, both arrangements are possible with the given clues.

Wait, let me reconsider more carefully. I have:
- Alice: dog (derived from elimination)
- Bob: cat or fish
- Carol: fish or cat

All three clues are satisfied with Alice having the dog. For Bob and Carol, I need to check if there are any additional constraints.

Looking at the clues again, there are no further restrictions on whether Bob has the cat vs. fish, or Carol has the fish vs. cat.

However, let me verify by checking both possibilities:

Possibility 1: Bob has cat, Carol has fish
- Alice does not have the cat ✓ (Alice has dog)
- Bob does not have the dog ✓ (Bob has cat)  
- The person with the fish is not Alice ✓ (Carol has fish)

Possibility 2: Bob has fish, Carol has cat
- Alice does not have the cat ✓ (Alice has dog)
- Bob does not have the dog ✓ (Bob has fish)
- The person with the fish is not Alice ✓ (Bob has fish)

Both possibilities satisfy all constraints. Let me see if I missed something...

Actually, both solutions work with the given constraints. The problem appears to be underdetermined. 

But if I must give one answer, both are valid:

**Solution:**
- Alice has the dog
- Bob has either the cat or the fish
- Carol has either the fish or the cat (whichever Bob doesn't have)

The constraints don't uniquely determine whether Bob has the cat or fish.

In [133]:
# TIER 4 anthropic/claude-3-haiku
display(
    Markdown(
        generate_completion(
        model_id="anthropic/claude-3-haiku", 
        messages=prompt, 
        #disable_reasoning=True # nonreasoning model does not need the parameter
        )
    )
)

To solve this problem, we can use the process of elimination and logical reasoning.

Given information:
- Alice, Bob, and Carol each have a different pet: a cat, a dog, and a fish.
- Alice does not have the cat.
- Bob does not have the dog.
- The person with the fish is not Alice.

Step 1: Determine who has the cat.
Since Alice does not have the cat, that means either Bob or Carol has the cat.

Step 2: Determine who has the dog.
Since Bob does not have the dog, that means either Alice or Carol has the dog.

Step 3: Determine who has the fish.
Since the person with the fish is not Alice, that means either Bob or Carol has the fish.

Combining the information from steps 1, 2, and 3, we can conclude the following:
- Alice has the dog.
- Bob has the cat.
- Carol has the fish.

Reasoning:
- Alice does not have the cat, so it must be Bob.
- Bob does not have the dog, so it must be Alice.
- The person with the fish is not Alice, so it must be Carol.

Therefore, the pets are distributed as follows:
- Alice has the dog.
- Bob has the cat.
- Carol has the fish.

In [95]:
## Tier 1 - openai/o1
display(
    Markdown(
        generate_completion(
        model_id="openai/o1", 
        messages=prompt
        )
    )
)

Answer:  
Alice must have the dog, but there is not enough information to determine which of Bob or Carol has the cat or the fish.

Explanation step by step:

1) Rule out pets for Alice:  
   • “Alice does not have the cat.”  
   • “The person with the fish is not Alice.”  
   Together, these mean Alice cannot have the cat or the fish, so she must have the dog.

2) Rule out pets for Bob:  
   • “Bob does not have the dog.”  
   This is automatically satisfied once Alice is determined to have the dog; Bob therefore could have either the cat or the fish.

3) Who has the fish?  
   Since the fish is not with Alice, it must be with Bob or Carol. But the puzzle gives no further constraint on whether Bob or Carol has the fish (and hence who has the cat).

Thus, the only certainty is that Alice has the dog. Bob and Carol must between them have the cat and the fish, in either order. No unique assignment of cat or fish to Bob or Carol can be deduced from the given clues.

---
## Demo 2 — Math Word Problem

Find all valid integer combinations. Traditional models often miss cases or give wrong totals.

In [136]:
content = """
A store sells apples for $0.75 each and oranges for $1.10 each.
Sarah buys some apples and oranges and pays exactly $10.00 total.
She buys at least one of each fruit.

What are ALL possible combinations she could have bought?
"""
prompt = [{"role": "user", "content": content}]

In [135]:
## Tier 1 - deepseek-r1 | MINIMAL reasoning efforts
display(
    Markdown(
        generate_completion(
        model_id="deepseek/deepseek-r1", 
        messages=prompt,
        disable_reasoning=False, # cannot set as True
        reasoning_effort= "minimal"
        )
    )
)

Sarah buys apples for $0.75 each and oranges for $1.10 each, paying exactly $10.00 total, with at least one of each fruit.

The equation for the total cost is:  
\[0.75a + 1.10o = 10.00\]  
where \(a\) is the number of apples and \(o\) is the number of oranges.

Converting to cents to avoid decimals:  
\[75a + 110o = 1000\]

Dividing by 5 to simplify:  
\[15a + 22o = 200\]

Solving for integer solutions where \(a \geq 1\) and \(o \geq 1\):  
- \(o\) must satisfy \(o \equiv 5 \pmod{15}\) based on the congruence \(7o \equiv 5 \pmod{15}\).  
- Possible values of \(o\) are 5, 20, 35, etc., but \(o \leq 8\) (since \(22o \leq 185\) when \(a \geq 1\)) and \(o \geq 1\), so only \(o = 5\) is feasible.  
- Substituting \(o = 5\):  
  \[15a + 22 \times 5 = 200\]  
  \[15a + 110 = 200\]  
  \[15a = 90\]  
  \[a = 6\]

Verifying the cost:  
- 6 apples: \(6 \times 0.75 = 4.50\)  
- 5 oranges: \(5 \times 1.10 = 5.50\)  
- Total: \(4.50 + 5.50 = 10.00\)

No other values of \(o\) from 1 to 9 yield an integer \(a\) that satisfies the equation and constraints. Similarly, solving for \(a\) gives \(a \equiv 6 \pmod{22}\), and only \(a = 6\) is feasible within the range.

Thus, the only possible combination is 6 apples and 5 oranges.

\boxed{6} \text{ apples and } \boxed{5} \text{ oranges}

In [137]:
## Tier 1 - deepseek-r1 | HIGH reasoning efforts
display(
    Markdown(
        generate_completion(
        model_id="deepseek/deepseek-r1", 
        messages=prompt,
        disable_reasoning=False, # cannot set as True
        reasoning_effort= "high"
        )
    )
)

Sarah buys apples for $0.75 each and oranges for $1.10 each, paying exactly $10.00 total, with at least one of each fruit.

The equation for the total cost is:  
$0.75A + 1.10O = 10.00$, where $A$ is the number of apples and $O$ is the number of oranges.

Converting to cents to avoid decimals:  
$75A + 110O = 1000$.

Dividing by the greatest common divisor of 5:  
$15A + 22O = 200$.

Solving for integer solutions with $A \geq 1$ and $O \geq 1$:  
- $A$ must satisfy $15A \equiv 2 \pmod{22}$, leading to $A \equiv 6 \pmod{22}$.  
- $O$ must satisfy $22O \equiv 5 \pmod{15}$, leading to $O \equiv 5 \pmod{15}$.

The general solution is:  
$A = 6 - 22k$  
$O = 5 + 15k$  
for integer $k$.

Considering the constraints $A \geq 1$ and $O \geq 1$:  
- For $k = 0$, $A = 6$ and $O = 5$.  
- For $k = 1$, $A = -16$ (invalid, negative).  
- For $k = -1$, $O = -10$ (invalid, negative).  

Thus, the only valid solution is $k = 0$, giving $A = 6$ and $O = 5$.

Verification:  
Cost of apples: $6 \times 0.75 = 4.50$  
Cost of oranges: $5 \times 1.10 = 5.50$  
Total: $4.50 + 5.50 = 10.00$.

No other combinations satisfy the conditions. Therefore, the only possible combination is **6 apples and 5 oranges**.

\boxed{(6, 5)}

In [138]:
## Tier 3 - deepseek-v4-flash
display(
    Markdown(
        generate_completion(
        model_id="deepseek/deepseek-v4-flash", 
        messages=prompt
        )
    )
)

Sarah bought 6 apples and 5 oranges.

\[
6 \times 0.75 + 5 \times 1.10 = 4.50 + 5.50 = 10.00
\]

This is the only combination that satisfies the conditions.

\boxed{6 \text{ apples and } 5 \text{ oranges}}

---
## Demo 3 —  Monty Hall Variant (5 doors!)

In [120]:
content = """
You're on a game show with 5 doors. 
One has a car. You pick Door 1. 
The host (who knows where the car is) opens two doors revealing goats. 
Should you switch, and if so, what's your new probability of winning?
"""
prompt = [{"role": "user", "content": content}]

In [122]:
## Tier 3: gpt-4o
display(
    Markdown(
        generate_completion(
        model_id="openai/gpt-4o", 
        messages=prompt
        )
    )
)

Yes, you should switch. 

This scenario is a variation of the classic "Monty Hall problem." When you first chose Door 1, the probability of the car being behind that door was 1/5, and the probability of the car being behind one of the other four doors was 4/5. 

After you choose Door 1, the host, who knows where the car is, opens two other doors, both revealing goats. This act does not affect the initial probability of 1/5 for Door 1. Instead, it redistributes the initial 4/5 probability among the remaining unchosen doors. Since the host knows where the car is and opens doors specifically not revealing the car, switching means choosing between fewer doors with a redistributed probability. 

Now, only two doors are left unopened: Door 1 (your original choice) and one other door. The probability that the car is behind that other door becomes 4/5, as you effectively have simplified the problem to two remaining options, with the probabilities still reflecting the initial distribution but focused between your first choice and the single remaining unopened door. 

Thus, by switching your choice to the remaining unopened door, your probability of winning the car increases to 4/5 or 80%.

In [123]:
## Tier 1: openai o1 model
display(
    Markdown(
        generate_completion(
        model_id="openai/o1", 
        messages=prompt
        )
    )
)

A succinct way to see that switching is better:

1. Your chance of having chosen the car initially is 1/5.  
2. Hence the chance the car is among the other four doors is 4/5.  
3. After you choose Door 1, the host, knowing where the car is, opens two goat doors among those other four doors. This leaves two unopened doors besides your original choice.  
4. If your original choice is wrong (probability 4/5), exactly one of those two remaining doors must have the car. Therefore, if you switch and choose randomly between them, you have a 1/2 chance of selecting the car in that scenario.  
5. Combining these: your total probability of winning by switching is (4/5) × (1/2) = 2/5 = 40 percent, which is better than the 1/5 = 20 percent chance if you stick with your first door.  

So yes—you should switch, and your probability of winning improves to 40 percent.

---
## Demo 4 - Expected Tosses for Consecutive Heads 

In [124]:
content = """
You flip a fair coin repeatedly. 
What is the expected number of total flips required to get two consecutive heads (HH)? 
What about three consecutive heads (HHH)?
"""
prompt = [{"role": "user", "content": content}]

In [129]:
## tier 2 - llma-4-maverick
display(
    Markdown(
        generate_completion(
        model_id="meta-llama/llama-4-maverick", 
        messages=prompt
        )
    )
)

## Step 1: Understand the problem and define the approach
To solve this, we need to calculate the expected number of flips to get two consecutive heads (HH) and then extend the reasoning to get three consecutive heads (HHH). We'll use the concept of expected value and recursive relations to model the problem.

## Step 2: Calculate the expected number of flips to get HH
Let's denote $E_{HH}$ as the expected number of flips to get two consecutive heads. We start by considering the possible outcomes of the first few flips and how they affect $E_{HH}$. If the first flip is T (tail), we are back to the starting point but have used 1 flip. If the first flip is H and the second is T, we again reset to the starting point but have used 2 flips. If the first two flips are HH, we achieve our goal in 2 flips.

## Step 3: Set up the equation for $E_{HH}$
- The probability of getting T on the first flip is $\frac{1}{2}$, and we're back to the start, so the contribution to $E_{HH}$ is $\frac{1}{2}(1 + E_{HH})$.
- The probability of getting H followed by T is $\frac{1}{4}$, and again, we're back to the start, contributing $\frac{1}{4}(2 + E_{HH})$.
- The probability of getting HH is $\frac{1}{4}$, and we stop, contributing $\frac{1}{4}(2)$.
So, the equation becomes: $E_{HH} = \frac{1}{2}(1 + E_{HH}) + \frac{1}{4}(2 + E_{HH}) + \frac{1}{4}(2)$.

## Step 4: Solve the equation for $E_{HH}$
Let's simplify and solve for $E_{HH}$:
$E_{HH} = \frac{1}{2} + \frac{1}{2}E_{HH} + \frac{1}{2} + \frac{1}{4}E_{HH} + \frac{1}{2}$.
Combining like terms: $E_{HH} = \frac{1}{2}E_{HH} + \frac{1}{4}E_{HH} + \frac{1}{2} + \frac{1}{2} + \frac{1}{2}$.
This simplifies to: $E_{HH} = \frac{3}{4}E_{HH} + \frac{3}{2}$.
Subtracting $\frac{3}{4}E_{HH}$ from both sides gives: $\frac{1}{4}E_{HH} = \frac{3}{2}$.
Multiplying both sides by 4 gives: $E_{HH} = 6$.

## Step 5: Calculate the expected number of flips to get HHH
Let's denote $E_{HHH}$ as the expected number of flips to get three consecutive heads. We extend the logic from $E_{HH}$.
- If the first flip is T, we have $1 + E_{HHH}$.
- If the first flip is H and the second is T, we have $2 + E_{HHH}$.
- If the first two flips are HH and the third is T, we have $3 + E_{HHH}$.
- If the first three flips are HHH, we have $3$.

## Step 6: Set up the equation for $E_{HHH}$
The equation becomes: $E_{HHH} = \frac{1}{2}(1 + E_{HHH}) + \frac{1}{4}(2 + E_{HHH}) + \frac{1}{8}(3 + E_{HHH}) + \frac{1}{8}(3)$.

## Step 7: Solve the equation for $E_{HHH}$
Let's simplify and solve for $E_{HHH}$:
$E_{HHH} = \frac{1}{2} + \frac{1}{2}E_{HHH} + \frac{1}{2} + \frac{1}{4}E_{HHH} + \frac{3}{8} + \frac{1}{8}E_{HHH} + \frac{3}{8}$.
Combining like terms: $E_{HHH} = (\frac{1}{2} + \frac{1}{4} + \frac{1}{8})E_{HHH} + (\frac{1}{2} + \frac{1}{2} + \frac{3}{8} + \frac{3}{8})$.
This simplifies to: $E_{HHH} = \frac{7}{8}E_{HHH} + (\frac{1}{2} + \frac{1}{2} + \frac{3}{4})$.
Simplifying further: $E_{HHH} = \frac{7}{8}E_{HHH} + (1 + \frac{3}{4}) = \frac{7}{8}E_{HHH} + \frac{7}{4}$.
Subtracting $\frac{7}{8}E_{HHH}$ from both sides gives: $\frac{1}{8}E_{HHH} = \frac{7}{4}$.
Multiplying both sides by 8 gives: $E_{HHH} = 14$.

The final answer is: $\boxed{6, 14}$

In [130]:
## tier 4: llama-3.2-3b
display(
    Markdown(
        generate_completion(
        model_id="meta-llama/llama-3.2-3b-instruct", 
        messages=prompt
        )
    )
)

To solve this problem, we can use the concept of a Markov chain or a recursive approach.

Let's define the expected number of flips required to get two consecutive heads (HH) as E2 and three consecutive heads (HHH) as E3.

For E2, we can break it down into two cases:

1. The first flip is a tail (T). In this case, we start over, so the expected number of flips required to get two consecutive heads is E2 + 1.
2. The first flip is a head (H). In this case, we need one more flip to get the second head, so the expected number of flips required to get two consecutive heads is E2 + 1.

We can write the equation for E

In [131]:
## tier 3 grok-4.3
display(
    Markdown(
        generate_completion(
        model_id="x-ai/grok-4.3", 
        messages=prompt
        )
    )
)

The expected number of flips to reach a given sequence of heads can be found using a finite-state Markov chain whose states track the current overlap (i.e., the longest proper prefix of the target sequence that is also a suffix of the flips observed so far).  An auxiliary absorbing state records that the target has been reached.  Let \(E_i\) denote the expected number of additional flips needed when the chain is in state \(i\).  Because the coin is fair, each transition equation is obtained by conditioning on the outcome of a single flip and adding the one flip that was just performed.

For the target HH the states are  
- state 0: no consecutive head yet,  
- state 1: the last flip was H,  
- state 2: HH has occurred (absorbing).  

The resulting system is
\[
E_0=1+\tfrac12 E_1+\tfrac12 E_0,\qquad
E_1=1+\tfrac12\cdot0+\tfrac12 E_0.
\]
Substitution yields the unique solution \(E_0=6\).  The process begins in state 0, therefore six flips are expected.

For the target HHH the states become  
- state 0: no head yet,  
- state 1: ends with exactly one H,  
- state 2: ends with exactly HH,  
- state 3: ends with HHH (absorbing).  

The system is now
\[
E_0=1+\tfrac12 E_1+\tfrac12 E_0,
\quad
E_1=1+\tfrac12 E_2+\tfrac12 E_0,
\quad
E_2=1+\tfrac12\cdot0+\tfrac12 E_0.
\]
Back-substitution produces the unique solution \(E_0=14\).  Again the process begins in state 0, so fourteen flips are expected.

(The same linear-algebraic technique works for any finite pattern on a sequence of i.i.d. trials; each additional required head simply adds one more transient state.)

Final answer: 6; 14

# Demo 5 - Solving Dice Problems with Markov Chains

In [140]:
content = """
If you roll a die continuously, are you more likely to roll two consecutive 5’s or a 5 immediately followed by a 6? 
On average how many rolls are required to roll two consecutive 5’s? How about rolling 5 immediately followed by a 6
"""
prompt = [{"role": "user", "content": content}]

In [141]:
## tier 4 anthropic/claude-3-haiku
display(
    Markdown(
        generate_completion(
        model_id="anthropic/claude-3-haiku", 
        messages=prompt
        )
    )
)

To determine the likelihood of rolling two consecutive 5's or a 5 immediately followed by a 6, we need to consider the probability of each event.

1. Rolling two consecutive 5's:
   - The probability of rolling a 5 on any given roll is 1/6.
   - The probability of rolling two consecutive 5's is the product of the probabilities of rolling a 5 on the first roll and a 5 on the second roll, which is (1/6) × (1/6) = 1/36.

2. Rolling a 5 immediately followed by a 6:
   - The probability of rolling a 5 on the first roll is 1/6.
   - The probability of rolling a 6 on the second roll is 1/6.
   - The probability of rolling a 5 immediately followed by a 6 is the product of the probabilities of rolling a 5 and then a 6, which is (1/6) × (1/6) = 1/36.

Therefore, the probability of rolling two consecutive 5's and the probability of rolling a 5 immediately followed by a 6 are both 1/36, which are equally likely.

Now, let's calculate the average number of rolls required to achieve these events:

1. Average number of rolls to roll two consecutive 5's:
   - Since the probability of rolling two consecutive 5's is 1/36, the average number of rolls required to achieve this event is 36.

2. Average number of rolls to roll a 5 immediately followed by a 6:
   - Since the probability of rolling a 5 immediately followed by a 6 is 1/36, the average number of rolls required to achieve this event is also 36.

In summary, the probability of rolling two consecutive 5's and the probability of rolling a 5 immediately followed by a 6 are both 1/36, and the average number of rolls required to achieve either event is 36.

In [143]:
## tier 3 google/gemini-2.5-flash
display(
    Markdown(
        generate_completion(
        model_id="google/gemini-2.5-flash", 
        messages=prompt
        )
    )
)

Let's break this down into two parts: comparing their likelihood and calculating the average number of rolls.

### Part 1: Are you more likely to roll two consecutive 5's or a 5 immediately followed by a 6?

The probability of rolling a specific number on a fair six-sided die is $1/6$. Rolls are independent events.

*   **For two consecutive 5's (5, 5):**
    *   Probability of rolling a 5 on the first roll: $1/6$
    *   Probability of rolling a 5 on the second roll: $1/6$
    *   Probability of (5 then 5) = $(1/6) \times (1/6) = 1/36$

*   **For a 5 immediately followed by a 6 (5, 6):**
    *   Probability of rolling a 5 on the first roll: $1/6$
    *   Probability of rolling a 6 on the second roll: $1/6$
    *   Probability of (5 then 6) = $(1/6) \times (1/6) = 1/36$

**Conclusion for Part 1:** They are **equally likely** for any given pair of consecutive rolls. In other words, if you pick any two moments in time, the chance of seeing `55` is the same as seeing `56`.

### Part 2: On average how many rolls are required?

This is where the "overlap" property of the sequence comes into play and causes a difference. We can solve this using the method of expected values (or absorbed Markov chains). Let $E$ be the expected number of rolls needed.

#### For two consecutive 5's (55):

Let $E_{55}$ be the expected number of rolls to get `55`.
Let $E_0$ be the expected number of rolls from a state where no part of the desired sequence has been made (or has been broken). This is our target $E_{55}$.
Let $E_1$ be the expected number of rolls from a state where the *last* roll was a 5, but not immediately preceded by another 5.

*   **From $E_0$ (no progress/broken sequence):**
    *   If you roll a 5 (probability $1/6$), you move to state $E_1$.
    *   If you roll any other number (1, 2, 3, 4, 6) (probability $5/6$), you stay in state $E_0$.
    *   So, $E_0 = 1 + (1/6)E_1 + (5/6)E_0$

*   **From $E_1$ (last roll was a 5):**
    *   If you roll a 5 (probability $1/6$), you achieve `55` (success!). The expected rolls from here is 0.
    *   If you roll any other number (1, 2, 3, 4, 6) (probability $5/6$), you break the sequence and return to state $E_0$.
    *   So, $E_1 = 1 + (1/6) \times 0 + (5/6)E_0 = 1 + (5/6)E_0$

Now, substitute $E_1$ into the equation for $E_0$:
$E_0 = 1 + (1/6)(1 + (5/6)E_0) + (5/6)E_0$
$E_0 = 1 + 1/6 + (5/36)E_0 + (30/36)E_0$
$E_0 = 7/6 + (35/36)E_0$
$E_0 - (35/36)E_0 = 7/6$
$(1/36)E_0 = 7/6$
$E_0 = (7/6) \times 36 = 7 \times 6 = 42$

On average, **42 rolls** are required to roll two consecutive 5's.

#### For a 5 immediately followed by a 6 (56):

Let $E_{56}$ be the expected number of rolls to get `56`.
Let $E'_0$ be the expected number of rolls from a state where no part of the desired sequence has been made (or has been broken). This is our target $E_{56}$.
Let $E'_1$ be the expected number of rolls from a state where the *last* roll was a 5.

*   **From $E'_0$ (no progress/broken sequence):**
    *   If you roll a 5 (probability $1/6$), you move to state $E'_1$.
    *   If you roll any other number (1, 2, 3, 4, 6) (probability $5/6$), you stay in state $E'_0$.
    *   So, $E'_0 = 1 + (1/6)E'_1 + (5/6)E'_0$

*   **From $E'_1$ (last roll was a 5):**
    *   If you roll a 6 (probability $1/6$), you achieve `56` (success!). Expected rolls from here is 0.
    *   If you roll a 5 (probability $1/6$), you are still in state $E'_1$ (the last roll is still a 5).
    *   If you roll any other number (1, 2, 3, 4) (probability $4/6$), you break the sequence and return to state $E'_0$.
    *   So, $E'_1 = 1 + (1/6) \times 0 + (1/6)E'_1 + (4/6)E'_0$
    *   $E'_1 = 1 + (1/6)E'_1 + (4/6)E'_0$
    *   $(5/6)E'_1 = 1 + (4/6)E'_0$
    *   $E'_1 = 6/5 + (4/5)E'_0$

Now, substitute $E'_1$ into the equation for $E'_0$:
$E'_0 = 1 + (1/6)(6/5 + (4/5)E'_0) + (5/6)E'_0$
$E'_0 = 1 + 1/5 + (4/30)E'_0 + (25/30)E'_0$
$E'_0 = 6/5 + (29/30)E'_0$
$E'_0 - (29/30)E'_0 = 6/5$
$(1/30)E'_0 = 6/5$
$E'_0 = (6/5) \times 30 = 6 \times 6 = 36$

On average, **36 rolls** are required to roll a 5 immediately followed by a 6.

### Summary and Intuition for the Difference:

*   **Likelihood:** Both are equally likely for any given pair of rolls (1/36).
*   **Average Rolls:**
    *   For **55**: 42 rolls
    *   For **56**: 36 rolls

This means you are more likely to achieve "5 immediately followed by 6" *sooner* (in fewer average rolls) than "two consecutive 5's."

**Why the difference?**
The key lies in how "failed" rolls affect your progress:

1.  **For 55:** If you're waiting for `55`, and you roll a `5` (partial success), but then the *next* roll is anything other than a `5` (e.g., a `6`), you lose all your progress. You have to start over by waiting for a new `5`.
2.  **For 56:** If you're waiting for `56`, and you roll a `5` (partial success), what if the next roll is *another `5`*? You haven't completed `56`, but you also haven't lost *all* your progress. You're still in the state where your *last* roll was a `5`, so you're still one step away from completing `56` if the next roll is a `6`. This "self-preserving partial progress" (or the lack of a "dead end" when rolling a `5` again for `56`) makes it, on average, take less time to complete the `56` sequence compared to `55`.

This is a classic result in probability theory: patterns that can "overlap" with themselves (like `55` having the `5` as both prefix and suffix) generally take longer to appear than patterns that do not (like `56`).

In [144]:
## tier 2 meta-llama/llama-4-maverick
display(
    Markdown(
        generate_completion(
        model_id="meta-llama/llama-4-maverick", 
        messages=prompt
        )
    )
)

## Step 1: Understand the problem and the basic probabilities involved.
The problem asks us to compare the likelihood of rolling two consecutive 5's versus rolling a 5 immediately followed by a 6 when rolling a die continuously. It also asks for the average number of rolls required to achieve these outcomes. The probability of rolling any specific number on a fair six-sided die is 1/6.

## Step 2: Calculate the probability of rolling two consecutive 5's.
To roll two consecutive 5's, the probability is (1/6) * (1/6) = 1/36, because the rolls are independent.

## Step 3: Calculate the probability of rolling a 5 immediately followed by a 6.
The probability of rolling a 5 followed by a 6 is also (1/6) * (1/6) = 1/36, for the same reason as in Step 2.

## Step 4: Compare the probabilities of the two sequences.
Both sequences (5 followed by 5, and 5 followed by 6) have the same probability of 1/36. This means that, probabilistically, they are equally likely to occur in any given pair of rolls.

## Step 5: Understand that the question about likelihood is related to the probability of the sequences, not the average number of rolls to achieve them.
Since both sequences have the same probability (1/36), they are equally likely to occur in a pair of rolls.

## Step 6: Calculate the average number of rolls required to roll two consecutive 5's.
Let's denote the expected number of rolls to get two consecutive 5's as $E$. The probability of rolling a 5 on the first roll is 1/6. If this happens, we have a "partial success" and need to roll another 5. If we succeed on the second roll (with probability 1/6), we're done. If we fail (with probability 5/6), we're back to the starting point but having used 2 rolls. If the first roll is not a 5 (with probability 5/6), we're also back to the starting point but having used 1 roll. So, we can set up the equation: $E = (1/6)(1/6)(2) + (1/6)(5/6)(E+2) + (5/6)(E+1)$.

## Step 7: Simplify the equation for $E$.
$E = (1/36)(2) + (5/36)(E+2) + (5/6)(E+1)$.
$E = 2/36 + (5/36)E + 10/36 + (5/6)E + 5/6$.
$E = (5/36 + 5/6)E + (2 + 10)/36 + 5/6$.
$E = (5/36 + 30/36)E + 12/36 + 30/36$.
$E = (35/36)E + 42/36$.
$E - (35/36)E = 42/36$.
$(1/36)E = 42/36$.
$E = 42$.

## Step 8: Calculate the average number of rolls required to roll a 5 immediately followed by a 6.
Let's denote this expected value as $F$. The reasoning is similar, but now we consider the outcomes after the first roll. If the first roll is a 5 (with probability 1/6), and the second roll is a 6 (with probability 1/6), we're done with 2 rolls. If the first roll is a 5 but the second is not a 6 (with probability 5/6), we're back to a state where we've used 2 rolls but need to start over, except we have the information that the last roll was not a 6. However, the key insight is that after any roll that is not part of the sequence (or a 5 that is not followed by a 6), we essentially start over. The correct way to frame this is through a Markov chain or a similar recursive relation. Let's directly apply the formula considering the probabilities and states. $F = 1 + (5/6)E' + (1/6)(1/6)(1) + (1/6)(5/6)(1 + F)$, where $E'$ is the expected number of rolls to get a 5.

## Step 9: Simplify $E'$ and then $F$.
First, let's simplify $E'$. $E' = 1 + (5/6)E'$, so $E' = 6$. Now, substituting $E'$ into the equation for $F$: $F = 1 + (5/6)(6) + (1/36)(1) + (5/36)(1 + F)$.
$F = 1 + 5 + 1/36 + (5/36) + (5/36)F$.
$F = 6 + 6/36 + (5/36)F$.
$F = 6 + 1/6 + (5/36)F$.
$F = 37/6 + (5/36)F$.
$F - (5/36)F = 37/6$.
$(31/36)F = 37/6$.
$F = (37/6) * (36/31)$.
$F = 222/31$.
$F \approx 7.16$.

The final answer is: $\boxed{42}$

In [145]:
## tier 1 openai/o4-mini
display(
    Markdown(
        generate_completion(
        model_id="openai/o4-mini", 
        messages=prompt
        )
    )
)

If you’re looking for which “pattern” – 55 or 56 – shows up first in a long run of fair‐die rolls, it turns out that

  • P(“56 appears before 55”) = ½  =  P(“55 appears before 56”).  

However the two patterns do not have the same *waiting-time* distributions.  In particular

  • The expected number of rolls until the first occurrence of “55” is  
     E(55) = 42.  
  • The expected number of rolls until the first occurrence of “56” is  
     E(56) = 36.  

So although neither is more likely to beat the other (each has probability ½ of coming first), you will on average see a 5→6 sooner (36 rolls) than two 5’s in a row (42 rolls).

— Derivation sketch —

Label states by “how much of your target you have matched so far.”  For “55”:

  Let E₀ = expected additional rolls from scratch,  
      E₁ = expected additional rolls if your last roll was a 5 (so you’re one step from “55”).

  From state 0: on the next roll with prob. 1/6 you go to state 1; with prob. 5/6 you stay in 0.  
    E₀ = 1 + (1/6)·E₁ + (5/6)·E₀  
    ⇒ (1/6)E₀ = 1 + (1/6)E₁  ⇒  E₀ = 6 + E₁.

  From state 1: on the next roll with prob. 1/6 you complete “55” (0 further rolls), with prob. 5/6 you drop back to state 0.
    E₁ = 1 + (1/6)·0 + (5/6)·E₀  = 1 + (5/6)E₀.

  Substitute E₀ = 6 + E₁ into the second:
    E₁ = 1 + (5/6)(6 + E₁)
       = 1 + 5 + (5/6)E₁
       ⇒ (1 – 5/6)E₁ = 6
       ⇒ (1/6)E₁ = 6  ⇒  E₁ = 36.

  Then E₀ = 6 + 36 = 42.

Exactly the same setup for “56,” except that from state 1 (you’ve just rolled a 5) a 6 completes the pattern and any roll other than 5 or 6 breaks the match and sends you back to state 0.  Solving the two‐state equations in that case gives

  E₀ = 36  for “56.”

Finally, if you set up probabilities P₀, P₁ that “56 beats 55” starting from state 0 or 1, you find P₀ = P₁ = ½, so each pattern is equally likely to occur first.

#### correct answer:
consecutive 5’: 42
5 followed by 6: 36